# Fase 3 Projeto FIAP

### Modelagem e Manipulação de Dados

In [33]:
import pandas as pd
import numpy as np

# Data visualization library
import pygwalker as pyg

In [34]:
# Read and manipulation data

flights_df = pd.read_csv("../data/bronze/flights.csv", sep=",")
airlines_df = pd.read_csv("../data/bronze/airlines.csv", sep=",")
airports_df = pd.read_csv("../data/bronze/airports.csv", sep=",")


/var/folders/h0/9cxt3flx6px_mz8j5myq3cjw0000gn/T/ipykernel_39291/3539612395.py:3: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  flights_df = pd.read_csv("../data/bronze/flights.csv", sep=",")


-----

#### Manipulation Flights Database

In [35]:
flights_df.head(10)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,610.0,8.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,509.0,-17.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,753.0,-10.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,532.0,-13.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,656.0,-15.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
flights_df["DATE"] = pd.to_datetime(
    flights_df["YEAR"].astype(str) + "-" +
    flights_df["MONTH"].astype(str) + "-" +
    flights_df["DAY"].astype(str),
    format="%Y-%m-%d"
)

In [37]:
flights_df.drop(columns=[
    "YEAR",
    "MONTH",
    "DAY"
], inplace=True)

#### Manipulação nas colunas de HHMM, segue abaixo quais são:

- SCHEDULED_DEPARTURE: Horário de partida programado
- DEPARTURE_TIME: Horário real de partida
- WHEELS_OFF: Horário em que o avião decolou 
- WHEELS_ON: Horário em que as rodas tocaram o solo
- SCHEDULED_ARRIVAL: Horário de chegada programado 
- ARRIVAL_TIME: Horário de chegada real

In [38]:
# Excluido as colunas que não precisam:
flights_df.drop(columns=[
    "WHEELS_OFF",
    "WHEELS_ON"
], inplace=True)

In [39]:
# Colunas que possuem NA e o total:

columns_hhmm = [
    "SCHEDULED_DEPARTURE",
    "SCHEDULED_ARRIVAL",
    "DEPARTURE_TIME",
    "ARRIVAL_TIME"
]


for column in columns_hhmm:
    somatorio = flights_df[column].isna().sum()
    print(f"{column}: {somatorio}")

    if somatorio > 0:
        flights_df[column] = flights_df[column].fillna(0)
        flights_df[column] = flights_df[column].astype(int)

        print(f"Coluna {column} alterada para int, total NaN {flights_df[column].isna().sum()}")




SCHEDULED_DEPARTURE: 0
SCHEDULED_ARRIVAL: 0
DEPARTURE_TIME: 86153
Coluna DEPARTURE_TIME alterada para int, total NaN 0
ARRIVAL_TIME: 92513
Coluna ARRIVAL_TIME alterada para int, total NaN 0


In [40]:
flights_df["DISTANCE_KM"] = (flights_df["DISTANCE"] * 1.60934).round(0)

In [41]:

descricao_cancelamentos = {
    "A": "Airline",
    "B": "Weather",
    "C": "NAS",
    "D": "Security"
}

flights_df["CANCELLATION_DESCRIPTION"] = flights_df["CANCELLATION_REASON"].map(descricao_cancelamentos)

flights_df["CANCELLATION_DESCRIPTION"] = flights_df["CANCELLATION_DESCRIPTION"].fillna("N/A")

In [42]:
dias_semana = {
    1: "Segunda-feira",
    2: "Terça-feira",
    3: "Quarta-feira",
    4: "Quinta-feira",
    5: "Sexta-feira",
    6: "Sábado",
    7: "Domingo",
}

flights_df["WEEKDAY_NAME"] = flights_df["DAY_OF_WEEK"].map(dias_semana)



In [43]:
flights_df.head(10)

,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,...,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,DATE,DISTANCE_KM,CANCELLATION_DESCRIPTION,WEEKDAY_NAME
0,4,AS,98,N407AS,ANC,SEA,5,2354,-11.0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2330.0,N/A,Quinta-feira
1,4,AA,2336,N3KUAA,LAX,PBI,10,2,-8.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,3750.0,N/A,Quinta-feira
2,4,US,840,N171US,SFO,CLT,20,18,-2.0,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,3695.0,N/A,Quinta-feira
3,4,AA,258,N3HYAA,LAX,MIA,20,15,-5.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,3769.0,N/A,Quinta-feira
4,4,AS,135,N527AS,SEA,ANC,25,24,-1.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2330.0,N/A,Quinta-feira
5,4,DL,806,N3730B,SFO,MSP,25,20,-5.0,18.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2557.0,N/A,Quinta-feira
6,4,NK,612,N635NK,LAS,MSP,25,19,-6.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2091.0,N/A,Quinta-feira
7,4,US,2013,N584UW,LAX,CLT,30,44,14.0,13.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,3420.0,N/A,Quinta-feira
8,4,AA,1112,N3LAAA,SFO,DFW,30,19,-11.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2356.0,N/A,Quinta-feira
9,4,DL,1173,N826DN,LAS,ATL,30,33,3.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,2812.0,N/A,Quinta-feira


In [44]:
flights_df.shape[0]

5819079

----

#### Merge DFs

In [45]:
first_merge_dfs = flights_df.merge(
    airlines_df, 
    how="left", 
    left_on="AIRLINE", 
    right_on="IATA_CODE"
)

first_merge_dfs.drop(columns = ["IATA_CODE"], inplace=True)
first_merge_dfs.rename(columns={
    "AIRLINE_x": "AIRLANE_CODE",
    "AIRLINE_y": "AIRLINE_NAME"
    }, inplace=True
)

In [46]:
df_origin_airport = first_merge_dfs.merge(
    airports_df,
    how="left",
    left_on="ORIGIN_AIRPORT",
    right_on="IATA_CODE"
)

df_origin_airport.drop(columns=[
    "IATA_CODE",
    ], inplace=True)

df_origin_airport.rename(columns={
    "AIRPORT": "ORIGIN_AIRPORT_NAME",
    "CITY": "ORIGIN_CITY_NAME",
    "STATE": "ORIGIN_STATE_NAME",
    "COUNTRY": "ORIGIN_COUNTRY_NAME",
    "LATITUDE": "ORIGIN_LATITUDE_NAME",
    "LONGITUDE": "ORIGIN_LONGITUDE_NAME"
}, inplace=True)

In [47]:
df_origin_destiny = df_origin_airport.merge(
    airports_df,
    how="left",
    left_on="DESTINATION_AIRPORT",
    right_on="IATA_CODE"
)

df_origin_destiny.drop(columns=[
    "IATA_CODE",
    ], inplace=True)

df_origin_destiny.rename(columns={
    "AIRPORT": "DESTINATION_AIRPORT_NAME",
    "CITY": "DESTINATION_CITY_NAME",
    "STATE": "DESTINATION_STATE_NAME",
    "COUNTRY": "DESTINATION_COUNTRY_NAME",
    "LATITUDE": "DESTINATION_LATITUDE_NAME",
    "LONGITUDE": "DESTINATION_LONGITUDE_NAME"
}, inplace=True)

In [56]:
df_origin_destiny[["TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "ORIGIN_AIRPORT_NAME",
"ORIGIN_CITY_NAME",
"ORIGIN_STATE_NAME",
"ORIGIN_COUNTRY_NAME",
"ORIGIN_LATITUDE_NAME",
"ORIGIN_LONGITUDE_NAME","DESTINATION_AIRPORT_NAME",
"DESTINATION_CITY_NAME",
"DESTINATION_STATE_NAME",
"DESTINATION_COUNTRY_NAME",
"DESTINATION_LATITUDE_NAME",
"DESTINATION_LONGITUDE_NAME"]].head()

,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_CITY_NAME,ORIGIN_STATE_NAME,ORIGIN_COUNTRY_NAME,ORIGIN_LATITUDE_NAME,ORIGIN_LONGITUDE_NAME,DESTINATION_AIRPORT_NAME,DESTINATION_CITY_NAME,DESTINATION_STATE_NAME,DESTINATION_COUNTRY_NAME,DESTINATION_LATITUDE_NAME,DESTINATION_LONGITUDE_NAME
0,N407AS,ANC,SEA,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931
1,N3KUAA,LAX,PBI,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559
2,N171US,SFO,CLT,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313
3,N3HYAA,LAX,MIA,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056
4,N527AS,SEA,ANC,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619


In [58]:
airports_df.query(
    """
    IATA_CODE == 'ANC' | IATA_CODE == 'SEA' | IATA_CODE == 'LAX' | IATA_CODE == 'PBI' | IATA_CODE == 'SFO' | IATA_CODE == 'CLT' |  IATA_CODE == 'LAX' | IATA_CODE == 'MIA' 
    """
)

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
17,ANC,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619
66,CLT,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313
176,LAX,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807
203,MIA,Miami International Airport,Miami,FL,USA,25.79325,-80.29056
235,PBI,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559
277,SEA,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931
278,SFO,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484


----

#### Save Dataframe

In [60]:
df_origin_destiny.to_csv("../data/silver/flights_delay_cancelled.csv", index=False, sep=",")

In [61]:
teste = pd.read_csv("../data/silver/flights_delay_cancelled.csv", sep=",")
teste.shape

/var/folders/h0/9cxt3flx6px_mz8j5myq3cjw0000gn/T/ipykernel_39291/3860805405.py:1: DtypeWarning: Columns (4,5,31,32,33,34,37,38,39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  teste = pd.read_csv("../data/silver/flights_delay_cancelled.csv", sep=",")


(5819079, 43)